# 04 - Preparo de treino LSTM do Rio Doce

## Objetivo

Este notebook tem como objetivo analisar os artefatos gerados na Fase 3B do Estudo de Caso 2, validando a divisão temporal em treino, validação e teste, bem como a preparação final dos dados para treinamento da LSTM.

A execução oficial desta etapa é realizada via CLI por meio do comando:

`python -m src.river_level.run_preparar_treino --config experiments/river_level/baseline_lstm.yaml`

Assim, este notebook não substitui a etapa oficial do pipeline, servindo apenas como apoio para inspeção, validação e documentação dos resultados.

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [4]:
# 1. Aponta o Python para a pasta raiz do seu projeto MLOps
raiz_projeto = r"C:\Users\guerr\OneDrive\Documents\VSCode\FrameWork_MLOps_Unificado"

# Altera o diretório de trabalho apenas se já não estiver na raiz
if str(Path.cwd()) != raiz_projeto:
    os.chdir(raiz_projeto)

print(f"Buscando arquivos a partir de: {Path.cwd()}\n")

# 2. Define os caminhos relativos dos seus arquivos
caminho_dataset_preparado = Path("data/processed/river_level/dataset_treino_validacao_teste_baseline_lstm.npz")
caminho_scaler_X = Path("artifacts/river_level/baseline_lstm/preparo_treino/scaler_X_baseline_lstm.joblib")
caminho_scaler_y = Path("artifacts/river_level/baseline_lstm/preparo_treino/scaler_y_baseline_lstm.joblib")
caminho_resumo = Path("artifacts/river_level/baseline_lstm/preparo_treino/resumo_preparo_treino_baseline_lstm.json")

# 3. Checa se os arquivos existem (agora deve retornar True)
print(caminho_dataset_preparado.exists(), caminho_dataset_preparado)
print(caminho_scaler_X.exists(), caminho_scaler_X)
print(caminho_scaler_y.exists(), caminho_scaler_y)
print(caminho_resumo.exists(), caminho_resumo)

Buscando arquivos a partir de: C:\Users\guerr\OneDrive\Documents\VSCode\FrameWork_MLOps_Unificado

True data\processed\river_level\dataset_treino_validacao_teste_baseline_lstm.npz
True artifacts\river_level\baseline_lstm\preparo_treino\scaler_X_baseline_lstm.joblib
True artifacts\river_level\baseline_lstm\preparo_treino\scaler_y_baseline_lstm.joblib
True artifacts\river_level\baseline_lstm\preparo_treino\resumo_preparo_treino_baseline_lstm.json


## carregamento do dataset e do resumo

In [5]:
dataset_preparado = np.load(caminho_dataset_preparado, allow_pickle=True)

with open(caminho_resumo, "r", encoding="utf-8") as arquivo:
    resumo_preparo_treino = json.load(arquivo)

## visualização do resumo JSON

In [6]:
resumo_preparo_treino

{'coluna_alvo': 'Nivel',
 'indice_alvo_nas_entradas': 0,
 'passos_entrada': 180,
 'horizonte_previsao': 1,
 'tipo_scaler_X': 'minmax',
 'tipo_scaler_y': 'minmax',
 'proporcao_treino': 0.7,
 'proporcao_validacao': 0.15,
 'proporcao_teste': 0.15,
 'shape_X_treino': [5235, 180, 36],
 'shape_y_treino': [5235],
 'shape_X_validacao': [1121, 180, 36],
 'shape_y_validacao': [1121],
 'shape_X_teste': [1123, 180, 36],
 'shape_y_teste': [1123],
 'primeira_data_treino': '2005-09-28',
 'ultima_data_treino': '2020-01-27',
 'primeira_data_validacao': '2020-01-28',
 'ultima_data_validacao': '2023-02-21',
 'primeira_data_teste': '2023-02-22',
 'ultima_data_teste': '2026-03-20',
 'caminho_dataset_sequencial_entrada': 'data/processed/river_level/dataset_sequencial_baseline_lstm.npz',
 'caminho_dataset_saida': 'data/processed/river_level/dataset_treino_validacao_teste_baseline_lstm.npz',
 'caminho_scaler_X': 'artifacts/river_level/baseline_lstm/preparo_treino/scaler_X_baseline_lstm.joblib',
 'caminho_scal

## extração dos componentes principais

In [7]:
X_treino = dataset_preparado["X_treino"]
y_treino = dataset_preparado["y_treino"]

X_validacao = dataset_preparado["X_validacao"]
y_validacao = dataset_preparado["y_validacao"]

X_teste = dataset_preparado["X_teste"]
y_teste = dataset_preparado["y_teste"]

datas_alvo_treino = dataset_preparado["datas_alvo_treino"]
datas_alvo_validacao = dataset_preparado["datas_alvo_validacao"]
datas_alvo_teste = dataset_preparado["datas_alvo_teste"]

colunas_entrada = dataset_preparado["colunas_entrada"]
coluna_alvo = dataset_preparado["coluna_alvo"][0]
indice_alvo_nas_entradas = int(dataset_preparado["indice_alvo_nas_entradas"][0])
passos_entrada = int(dataset_preparado["passos_entrada"][0])
horizonte_previsao = int(dataset_preparado["horizonte_previsao"][0])

## conferência estrutural inicial

In [8]:
print("Shape X_treino:", X_treino.shape)
print("Shape y_treino:", y_treino.shape)

print("Shape X_validacao:", X_validacao.shape)
print("Shape y_validacao:", y_validacao.shape)

print("Shape X_teste:", X_teste.shape)
print("Shape y_teste:", y_teste.shape)

print("Quantidade de colunas de entrada:", len(colunas_entrada))
print("Coluna alvo:", coluna_alvo)
print("Índice do alvo nas entradas:", indice_alvo_nas_entradas)
print("Passos de entrada:", passos_entrada)
print("Horizonte de previsão:", horizonte_previsao)

Shape X_treino: (5235, 180, 36)
Shape y_treino: (5235,)
Shape X_validacao: (1121, 180, 36)
Shape y_validacao: (1121,)
Shape X_teste: (1123, 180, 36)
Shape y_teste: (1123,)
Quantidade de colunas de entrada: 36
Coluna alvo: Nivel
Índice do alvo nas entradas: 0
Passos de entrada: 180
Horizonte de previsão: 1


## conferência das datas das partições

In [9]:
print("Treino:", datas_alvo_treino[0], "até", datas_alvo_treino[-1])
print("Validação:", datas_alvo_validacao[0], "até", datas_alvo_validacao[-1])
print("Teste:", datas_alvo_teste[0], "até", datas_alvo_teste[-1])

Treino: 2005-09-28 até 2020-01-27
Validação: 2020-01-28 até 2023-02-21
Teste: 2023-02-22 até 2026-03-20


## conferência das quantidades e proporções

In [10]:
quantidade_total = len(y_treino) + len(y_validacao) + len(y_teste)

print("Quantidade total de amostras:", quantidade_total)
print("Treino:", len(y_treino), f"({len(y_treino) / quantidade_total:.4%})")
print("Validação:", len(y_validacao), f"({len(y_validacao) / quantidade_total:.4%})")
print("Teste:", len(y_teste), f"({len(y_teste) / quantidade_total:.4%})")

assert quantidade_total == 7479, "Quantidade total diferente da esperada."

Quantidade total de amostras: 7479
Treino: 5235 (69.9960%)
Validação: 1121 (14.9886%)
Teste: 1123 (15.0154%)


## checagem de NaN e infinito

In [11]:
print("NaN em X_treino:", np.isnan(X_treino).sum())
print("NaN em y_treino:", np.isnan(y_treino).sum())

print("NaN em X_validacao:", np.isnan(X_validacao).sum())
print("NaN em y_validacao:", np.isnan(y_validacao).sum())

print("NaN em X_teste:", np.isnan(X_teste).sum())
print("NaN em y_teste:", np.isnan(y_teste).sum())

print("Inf em X_treino:", np.isinf(X_treino).sum())
print("Inf em y_treino:", np.isinf(y_treino).sum())

print("Inf em X_validacao:", np.isinf(X_validacao).sum())
print("Inf em y_validacao:", np.isinf(y_validacao).sum())

print("Inf em X_teste:", np.isinf(X_teste).sum())
print("Inf em y_teste:", np.isinf(y_teste).sum())

assert np.isnan(X_treino).sum() == 0
assert np.isnan(y_treino).sum() == 0
assert np.isnan(X_validacao).sum() == 0
assert np.isnan(y_validacao).sum() == 0
assert np.isnan(X_teste).sum() == 0
assert np.isnan(y_teste).sum() == 0

assert np.isinf(X_treino).sum() == 0
assert np.isinf(y_treino).sum() == 0
assert np.isinf(X_validacao).sum() == 0
assert np.isinf(y_validacao).sum() == 0
assert np.isinf(X_teste).sum() == 0
assert np.isinf(y_teste).sum() == 0

NaN em X_treino: 0
NaN em y_treino: 0
NaN em X_validacao: 0
NaN em y_validacao: 0
NaN em X_teste: 0
NaN em y_teste: 0
Inf em X_treino: 0
Inf em y_treino: 0
Inf em X_validacao: 0
Inf em y_validacao: 0
Inf em X_teste: 0
Inf em y_teste: 0


## faixa dos valores escalados

In [12]:
print("Faixa X_treino:", float(X_treino.min()), "até", float(X_treino.max()))
print("Faixa y_treino:", float(y_treino.min()), "até", float(y_treino.max()))

print("Faixa X_validacao:", float(X_validacao.min()), "até", float(X_validacao.max()))
print("Faixa y_validacao:", float(y_validacao.min()), "até", float(y_validacao.max()))

print("Faixa X_teste:", float(X_teste.min()), "até", float(X_teste.max()))
print("Faixa y_teste:", float(y_teste.min()), "até", float(y_teste.max()))

Faixa X_treino: 0.0 até 1.0000001192092896
Faixa y_treino: 0.0 até 1.0
Faixa X_validacao: -0.10846781730651855 até 1.0
Faixa y_validacao: 0.04739004373550415 até 0.9369492530822754
Faixa X_teste: -0.0009660325013101101 até 1.829268217086792
Faixa y_teste: 0.011158451437950134 até 0.4805136024951935


## validação específica do treino

In [16]:
# Tolerância numérica para comparação com float32 após MinMaxScaler
tolerancia = 1e-5

assert X_treino.min() >= -tolerancia, f"Mínimo excedido: {X_treino.min()}"
assert X_treino.max() <= 1 + tolerancia, f"Máximo excedido: {X_treino.max()}"
assert y_treino.min() >= -tolerancia, f"Mínimo excedido: {y_treino.min()}"
assert y_treino.max() <= 1 + tolerancia, f"Máximo excedido: {y_treino.max()}"

print("Escalonamento do treino validado com sucesso.")

Escalonamento do treino validado com sucesso.


## validar os scalers e conferir a reversibilidade do alvo y
1. arregar os scalers

In [17]:
import joblib

scaler_X = joblib.load(caminho_scaler_X)
scaler_y = joblib.load(caminho_scaler_y)

print(type(scaler_X))
print(type(scaler_y))

<class 'sklearn.preprocessing._data.MinMaxScaler'>
<class 'sklearn.preprocessing._data.MinMaxScaler'>


## inspecionar parâmetros do scaler do alvo

In [18]:
print("min_ y:", scaler_y.data_min_)
print("max_ y:", scaler_y.data_max_)
print("range_ y:", scaler_y.data_range_)

min_ y: [1.22]
max_ y: [9.]
range_ y: [7.7799997]


## reverter alguns valores de y_treino

In [19]:
amostra_y_treino_escalado = y_treino[:5].reshape(-1, 1)
amostra_y_treino_original = scaler_y.inverse_transform(amostra_y_treino_escalado).reshape(-1)

print("y_treino escalado:", y_treino[:5])
print("y_treino revertido:", amostra_y_treino_original)

y_treino escalado: [0.3521851  0.3187661  0.28791773 0.28791773 0.25578406]
y_treino revertido: [3.96 3.7  3.46 3.46 3.21]


## carregar o dataset sequencial original da 3A para comparar

In [20]:
caminho_dataset_sequencial = Path("data/processed/river_level/dataset_sequencial_baseline_lstm.npz")
dataset_sequencial = np.load(caminho_dataset_sequencial, allow_pickle=True)

y_original_3a = dataset_sequencial["y"]
print("Shape y original 3A:", y_original_3a.shape)

Shape y original 3A: (7479,)


## comparar y_treino revertido com o início do y original

In [21]:
y_treino_revertido = scaler_y.inverse_transform(y_treino.reshape(-1, 1)).reshape(-1)

print("Primeiros 5 do y original 3A:", y_original_3a[:5])
print("Primeiros 5 do y treino revertido:", y_treino_revertido[:5])

assert np.allclose(y_treino_revertido, y_original_3a[:len(y_treino)], atol=1e-5)
print("Reversibilidade do alvo validada com sucesso.")

Primeiros 5 do y original 3A: [3.96 3.7  3.46 3.46 3.21]
Primeiros 5 do y treino revertido: [3.96 3.7  3.46 3.46 3.21]
Reversibilidade do alvo validada com sucesso.


In [22]:
dataset_preparado.close()
dataset_sequencial.close()  # se ele também estiver carregado
del dataset_preparado
del dataset_sequencial

## Conclusão da validação da Fase 3B

A etapa de preparo de treino da LSTM foi validada com sucesso.

Os resultados confirmaram que:

- o dataset final foi corretamente dividido em treino, validação e teste, preservando a ordem temporal;
- os tensores mantiveram o formato esperado para a LSTM;
- não foram encontrados valores ausentes ou infinitos nas partições;
- o `MinMaxScaler` foi ajustado apenas com a partição de treino;
- a faixa do treino ficou compatível com a normalização esperada, considerando pequena tolerância numérica associada ao uso de `float32`;
- as partições de validação e teste puderam apresentar valores fora do intervalo `[0, 1]`, o que é esperado quando surgem observações fora da faixa vista no treino;
- a transformação inversa do alvo confirmou a consistência entre a Fase 3A e a Fase 3B.

Dessa forma, a Fase 3B pode ser considerada concluída e o dataset gerado está apto para a próxima etapa de treinamento do modelo LSTM.